In [0]:
df = spark.read.table("workspace.default.lc_features")
print("Rows:", df.count(), "| Cols:", len(df.columns))
df.select("issue_year").groupBy("issue_year").count().orderBy("issue_year").show(20)

Rows: 1345349 | Cols: 20
+----------+------+
|issue_year| count|
+----------+------+
|      2007|   251|
|      2008|  1562|
|      2009|  4716|
|      2010| 11536|
|      2011| 21721|
|      2012| 53367|
|      2013|134804|
|      2014|223103|
|      2015|375545|
|      2016|293105|
|      2017|169321|
|      2018| 56318|
+----------+------+



In [0]:
train = df.filter(df.issue_year <= 2016)
test  = df.filter(df.issue_year >= 2017)

print("Train:", train.count(), "| Test:", test.count())
print("Train default rate:", train.agg({"is_bad": "avg"}).collect()[0][0])
print("Test default rate:",  test.agg({"is_bad": "avg"}).collect()[0][0])

Train: 1119710 | Test: 225639
Train default rate: 0.19697510962659975
Test default rate: 0.21291975234777677


In [0]:
from pyspark.sql.functions import col, when, log, sum as Fsum, count as Fcount

def compute_woe_iv(df, feature_col, target_col="is_bad"):
    # Count events (bad=1) and non-events (good=0) per bin
    agg = (df.groupBy(feature_col)
             .agg(Fsum(col(target_col)).alias("bad"),
                  (Fcount("*") - Fsum(col(target_col))).alias("good")))
    
    total_bad  = agg.agg(Fsum("bad")).collect()[0][0]
    total_good = agg.agg(Fsum("good")).collect()[0][0]
    
    # Laplace smoothing (+0.5) prevents ln(0) when a bin has zero of either class
    agg = agg.withColumn("pct_bad",  (col("bad")  + 0.5) / (total_bad  + 0.5))
    agg = agg.withColumn("pct_good", (col("good") + 0.5) / (total_good + 0.5))
    agg = agg.withColumn("woe", log(col("pct_good") / col("pct_bad")))
    agg = agg.withColumn("iv_contrib", (col("pct_good") - col("pct_bad")) * col("woe"))
    
    iv = agg.agg(Fsum("iv_contrib")).collect()[0][0]
    return agg.orderBy(feature_col), iv

inc_bin                        IV = 0.0288
dti_bin                        IV = 0.0727
util_bin                       IV = 0.0186
grade_num                      IV = 0.4677
home_ownership_idx             IV = 0.0265
purpose_idx                    IV = 0.0201
verification_status_idx        IV = 0.0532
issue_year                     IV = 0.0307
credit_history_months          IV = 0.0152
term_months                    IV = 0.1972
emp_length_years               SKIPPED (not in table)
has_public_record              IV = 0.0069
has_prior_delinq               IV = 0.0020


In [0]:
from pyspark.ml.feature import QuantileDiscretizer

# Drop the column if it's already there (from your earlier run)
if "cr_hist_bin" in train.columns:
    train = train.drop("cr_hist_bin")
if "cr_hist_bin" in test.columns:
    test = test.drop("cr_hist_bin")

qd = QuantileDiscretizer(numBuckets=5, inputCol="credit_history_months",
                         outputCol="cr_hist_bin", handleInvalid="keep")
qd_model = qd.fit(train)              # fit ONCE on train
train = qd_model.transform(train)
test  = qd_model.transform(test)      # reuse the trained model

In [0]:
features = [
    "inc_bin", "dti_bin", "util_bin",        # binned continuous
    "grade_num",                              # ordinal
    "home_ownership_idx", "purpose_idx",      # nominal indexed
    "verification_status_idx",
    "issue_year", "credit_history_months",    # date-derived (bin these if continuous)
    "term_months", "emp_length_years",
    "has_public_record", "has_prior_delinq"
]

iv_results = {}
woe_tables = {}
for f in features:
    if f in train.columns:
        woe_df, iv = compute_woe_iv(train, f)
        iv_results[f] = iv
        woe_tables[f] = woe_df
        print(f"{f:30s} IV = {iv:.4f}")
    else:
        print(f"{f:30s} SKIPPED (not in table)")

inc_bin                        IV = 0.0288
dti_bin                        IV = 0.0727
util_bin                       IV = 0.0186
grade_num                      IV = 0.4677
home_ownership_idx             IV = 0.0265
purpose_idx                    IV = 0.0201
verification_status_idx        IV = 0.0532
issue_year                     IV = 0.0307
credit_history_months          IV = 0.0152
term_months                    IV = 0.1972
emp_length_years               SKIPPED (not in table)
has_public_record              IV = 0.0069
has_prior_delinq               IV = 0.0020


In [0]:
def apply_woe(df, feature_col, woe_df):
    woe_map = woe_df.select(feature_col, "woe").withColumnRenamed("woe", f"{feature_col}_woe")
    return df.join(woe_map, on=feature_col, how="left")

# Keep features with IV >= 0.02 and < 0.5
keep = [f for f, iv in iv_results.items() if 0.02 <= iv < 0.5]
print("Keeping:", keep)

train_woe = train
test_woe  = test
for f in keep:
    train_woe = apply_woe(train_woe, f, woe_tables[f])
    test_woe  = apply_woe(test_woe,  f, woe_tables[f])  # use TRAIN's WoE map on test

Keeping: ['inc_bin', 'dti_bin', 'grade_num', 'home_ownership_idx', 'purpose_idx', 'verification_status_idx', 'issue_year', 'term_months']


In [0]:
print(train_woe.columns)

['term_months', 'issue_year', 'verification_status_idx', 'purpose_idx', 'home_ownership_idx', 'grade_num', 'dti_bin', 'inc_bin', 'is_bad', 'loan_amnt', 'int_rate', 'util_bin', 'credit_history_months', 'has_prior_delinq', 'has_public_record', 'annual_inc', 'dti', 'revol_util', 'grade', 'purpose', 'cr_hist_bin', 'inc_bin_woe', 'dti_bin_woe', 'grade_num_woe', 'home_ownership_idx_woe', 'purpose_idx_woe', 'verification_status_idx_woe', 'issue_year_woe', 'term_months_woe']


In [0]:
woe_cols = [f"{f}_woe" for f in keep]
final_cols = ["is_bad", "issue_year"] + woe_cols    # dropped "id"

train_woe.select(final_cols).write.mode("overwrite").saveAsTable("workspace.default.lc_train_woe")
test_woe.select(final_cols).write.mode("overwrite").saveAsTable("workspace.default.lc_test_woe")

print("Train cols:", len(woe_cols), "| Saved.")

Train cols: 8 | Saved.


In [0]:
expected = [f"{f}_woe" for f in keep]
missing = [c for c in expected if c not in train_woe.columns]
print("Missing WoE cols:", missing)

Missing WoE cols: []


In [0]:
spark.table("workspace.default.lc_train_woe").show(3)
spark.table("workspace.default.lc_train_woe").printSchema()
print("Train rows:", spark.table("workspace.default.lc_train_woe").count())
print("Test rows:",  spark.table("workspace.default.lc_test_woe").count())

+------+----------+--------------------+-------------------+------------------+----------------------+-------------------+---------------------------+------------------+------------------+
|is_bad|issue_year|         inc_bin_woe|        dti_bin_woe|     grade_num_woe|home_ownership_idx_woe|    purpose_idx_woe|verification_status_idx_woe|    issue_year_woe|   term_months_woe|
+------+----------+--------------------+-------------------+------------------+----------------------+-------------------+---------------------------+------------------+------------------+
|     0|      2013| 0.09609438834526296|0.05017973060616183|0.4841530699452569|   0.16794718568800557|0.20408835774908868|       -0.07039742135600495|0.2832761038100753|0.2890008758775992|
|     0|      2013| 0.09609438834526296|0.21279294168483012|0.4841530699452569|   0.16794718568800557|0.20408835774908868|         0.3765447754592887|0.2832761038100753|0.2890008758775992|
|     0|      2013|-0.20214517464775372|0.3730658618161

In [0]:
print(keep)

['inc_bin', 'dti_bin', 'grade_num', 'home_ownership_idx', 'purpose_idx', 'verification_status_idx', 'issue_year', 'term_months']
